# Non-Uniform Space-Filling for a 4-Input Carbon-Capture Example

This notebook builds a 10-run non-uniform space-filling (NUSF) design over a
4-input carbon-capture space (`G`, `lldg`, `w`, `L`). A per-candidate weight is
used to place more design points where the weight is higher. The notebook
compares maximum-weight-ratio (MWR) values of 2 and 5 under both the **Direct**
and **Ranked** weight-scaling methods.

The candidate set is `SDoE_CCSI_example.csv` (373 candidate points), which
includes a `CI Width Prior` column that is used as the weight.

## 1. Load and inspect the candidate set

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from idaes_sdoe import ColumnRoles, load_csv, prepare_design_setup
from idaes_sdoe.design import design_nonuniform
from idaes_sdoe.plotting import plot_nonuniform_weights, plot_pair_matrix
from idaes_sdoe.scaling import scale_weights

INPUTS = ["G", "lldg", "w", "L"]
WEIGHT = "CI Width Prior"
candidate = load_csv(Path("supporting_data/SDoE_CCSI_example.csv"))
print(f"{len(candidate)} candidate points; inputs {INPUTS}; weight '{WEIGHT}'")
candidate[INPUTS + [WEIGHT]].describe().loc[["min", "max"]]

Pairwise view of the four inputs (candidate set):

In [ ]:
plot_pair_matrix(candidate, columns=INPUTS, title="Carbon-capture candidate set")

## 2. Construct designs: Direct vs Ranked scaling, MWR 2 and 5

We build 10-run designs for MWR values of 2 and 5 under both scaling methods.
Direct scaling linearly maps the raw weights into ``[1, MWR]`` (preserving the
weight distribution's shape); Ranked scaling first replaces weights by their
ranks, giving a more even spread of scaled weights.

In [ ]:
NUM_RESTARTS = 30  # number of random starts

setup = prepare_design_setup(
    candidate=candidate,
    roles=ColumnRoles(inputs=INPUTS, weight=WEIGHT, index="Run"),
)
results = {}
for method in ["direct_mwr", "ranked_mwr"]:
    for mwr in [2, 5]:
        results[(method, mwr)] = design_nonuniform(
            setup=setup, design_size=10, num_restarts=NUM_RESTARTS, mwr=mwr,
            scale_method=method, random_state=1,
        )
        print(f"{method:11s} MWR={mwr}: criterion={results[(method, mwr)].criterion_value:.4f}")

## 3. CDBW plots: Direct vs Ranked

Comparing the CDBW plots highlights the difference between the two scaling
methods. Under **Direct** scaling the weight histogram keeps the original skewed
shape; under **Ranked** scaling the histogram is far more even (roughly equal
counts per bin).

In [ ]:
def cdbw(method, mwr):
    result = results[(method, mwr)]
    return plot_nonuniform_weights(
        result.scaled_design[INPUTS].to_numpy(),
        result.scaled_design[WEIGHT].to_numpy(),
        scale_weights(candidate[WEIGHT].to_numpy(), method=method, mwr=mwr),
        title=f"CDBW ({'Direct' if method == 'direct_mwr' else 'Ranked'} scaling, MWR = {mwr})",
    )

cdbw("direct_mwr", 2)

In [ ]:
cdbw("ranked_mwr", 2)

In [ ]:
cdbw("direct_mwr", 5)

In [ ]:
cdbw("ranked_mwr", 5)

Pairwise scatter of a selected design (Direct scaling, MWR = 2):

In [ ]:
plot_pair_matrix(results[("direct_mwr", 2)].design, columns=INPUTS,
                 candidate=candidate, title="NUSF design (Direct, MWR = 2)")

The choice of MWR sets how strongly the design concentrates near the
high-weight (near-optimum) region, and the scaling method controls how the raw
weights translate into that emphasis. Comparing designs across both settings
lets the experimenter pick the balance of space-filling and emphasis that best
fits the study.